In [ ]:
import torch
import torch.nn as nn

# ── Word embeddings ───────────────────────────────────────────────────
vocab_size = 10000   # number of unique words
embed_dim  = 128     # vector dimension for each word

embedding = nn.Embedding(vocab_size, embed_dim)

# Convert a sentence to a sequence of token IDs
# (in practice: use a tokenizer; here we simulate with random IDs)
sentence = ['He', 'saw', 'two', 'birds']
token_ids = torch.tensor([42, 1337, 7, 88])   # example IDs

# Look up embeddings — each word becomes a 128-d vector
word_vecs = embedding(token_ids)   # (4, 128)
print(f'Sentence length: {len(sentence)} words')
print(f'Embedding shape: {word_vecs.shape}')   # (4, 128)

# Words with similar meaning should have high cosine similarity
# (this only holds after training; here the embeddings are random)
import torch.nn.functional as F
sim = F.cosine_similarity(word_vecs[0:1], word_vecs[1:2])
print(f'Cosine similarity (random init): {sim.item():.3f}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ════════════════════════════════════════════════════════════════════
# ATTENTION MECHANISM — step by step, then as seq2seq
# ════════════════════════════════════════════════════════════════════

def attention(query, key, value, mask=None):
    """
    Scaled dot-product attention.
    query: (batch, n_q, d_k)
    key:   (batch, n_k, d_k)
    value: (batch, n_k, d_v)
    Returns: context (batch, n_q, d_v), weights (batch, n_q, n_k)
    """
    d_k = query.size(-1)

    # Step 1: alignment scores (dot product of query with every key)
    scores = torch.bmm(query, key.transpose(1, 2)) / math.sqrt(d_k)

    # Step 2: optional masking (set future positions to -inf)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    # Step 3: normalise to attention weights
    weights = F.softmax(scores, dim=-1)   # (batch, n_q, n_k) rows sum to 1

    # Step 4: weighted sum of values
    context = torch.bmm(weights, value)   # (batch, n_q, d_v)

    return context, weights

# ── Concrete example: translating 'Ich hasse' → 'I hate' ─────────────
batch, d = 1, 4

# Simulated encoder hidden states for ['Ich', 'hasse']
encoder_states = torch.randn(batch, 2, d)   # (1, 2, 4)

# Keys and Values come from encoder (through linear projections)
W_k = nn.Linear(d, d, bias=False)
W_v = nn.Linear(d, d, bias=False)
W_q = nn.Linear(d, d, bias=False)

keys   = W_k(encoder_states)   # (1, 2, 4)
values = W_v(encoder_states)   # (1, 2, 4)

# Query comes from decoder (current step's hidden state)
decoder_state = torch.randn(batch, 1, d)  # (1, 1, 4)
query = W_q(decoder_state)                # (1, 1, 4)

context, attn_weights = attention(query, keys, values)
print(f'Attention weights: {attn_weights[0, 0].detach().tolist()}')
# e.g. [0.73, 0.27] — 'Ich' gets most attention when decoding 'I'
print(f'Context vector:   {context.shape}')   # (1, 1, 4) — weighted sum

# ── Seq2Seq with attention ────────────────────────────────────────────
class Seq2SeqWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed       = nn.Embedding(vocab_size, embed_dim)
        self.encoder     = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder     = nn.LSTM(embed_dim + hidden_dim, hidden_dim,
                                   batch_first=True)
        # Projections for key, value, query
        self.W_k   = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_v   = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_q   = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def encode(self, src):
        emb  = self.embed(src)                   # (B, src_len, embed_dim)
        enc_out, (h, c) = self.encoder(emb)      # enc_out: (B, src_len, H)
        return enc_out, h, c

    def decode_step(self, tgt_token, h, c, enc_out):
        # Embed current target token
        emb = self.embed(tgt_token)              # (B, 1, embed_dim)

        # Compute attention over encoder outputs
        query   = self.W_q(h.transpose(0,1))     # (B, 1, H)
        keys    = self.W_k(enc_out)              # (B, src_len, H)
        values  = self.W_v(enc_out)              # (B, src_len, H)
        ctx, w  = attention(query, keys, values) # (B, 1, H)

        # Concatenate embedding with context vector, run decoder LSTM
        dec_input       = torch.cat([emb, ctx], dim=-1)
        dec_out, (h, c) = self.decoder(dec_input, (h, c))
        logits          = self.output(dec_out)   # (B, 1, vocab)
        return logits, h, c, w

    def forward(self, src, tgt):
        enc_out, h, c = self.encode(src)
        all_logits, all_weights = [], []
        for t in range(tgt.size(1)):
            logits, h, c, w = self.decode_step(
                tgt[:, t:t+1], h, c, enc_out)
            all_logits.append(logits)
            all_weights.append(w)
        logits  = torch.cat(all_logits, dim=1)   # (B, tgt_len, vocab)
        weights = torch.cat(all_weights, dim=1)  # (B, tgt_len, src_len)
        return logits, weights

model = Seq2SeqWithAttention(vocab_size=1000, embed_dim=64, hidden_dim=128)
src = torch.randint(0, 1000, (2, 5))   # 2 sentences, 5 src tokens
tgt = torch.randint(0, 1000, (2, 4))   # 4 target tokens
logits, weights = model(src, tgt)
print(f'Output logits:     {logits.shape}')   # (2, 4, 1000)
print(f'Attention weights: {weights.shape}')  # (2, 4, 5) src-tgt alignment

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ── CNN-LSTM Image Captioning with Spatial Attention ──────────────────
class ImageCaptioner(nn.Module):
    """
    Image captioning with spatial attention.
    Encoder: CNN extracts spatial feature map (H×W×D).
    Decoder: LSTM attends to spatial features at each word step.
    """
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512,
                 feat_dim=2048):
        super().__init__()

        # ── Visual encoder ────────────────────────────────────────────
        # ResNet50 backbone; remove final FC and pooling to keep spatial map
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])  # → (B,2048,7,7)
        self.feat_proj = nn.Linear(feat_dim, hidden_dim)   # project to hidden_dim

        # ── Attention ─────────────────────────────────────────────────
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)  # query from decoder
        self.W_k = nn.Linear(hidden_dim, hidden_dim, bias=False)  # key from visual
        self.W_v = nn.Linear(hidden_dim, hidden_dim, bias=False)  # value from visual

        # ── Language decoder ──────────────────────────────────────────
        self.embed   = nn.Embedding(vocab_size, embed_dim)
        # LSTM input: word embedding + attended visual context
        self.decoder = nn.LSTMCell(embed_dim + hidden_dim, hidden_dim)
        self.output  = nn.Linear(hidden_dim, vocab_size)

    def encode_image(self, image):
        feat = self.cnn(image)                       # (B, 2048, 7, 7)
        B, C, H, W = feat.shape
        feat = feat.permute(0, 2, 3, 1)              # (B, 7, 7, 2048)
        feat = feat.view(B, H*W, C)                  # (B, 49, 2048) — 49 spatial locs
        feat = self.feat_proj(feat)                  # (B, 49, hidden_dim)
        return feat

    def attend(self, h, visual_feat):
        query = self.W_q(h.unsqueeze(1))             # (B, 1, H)
        keys  = self.W_k(visual_feat)                # (B, 49, H)
        vals  = self.W_v(visual_feat)                # (B, 49, H)
        scores  = (query * keys).sum(-1)             # (B, 49) — dot product
        weights = torch.softmax(scores, dim=-1)      # (B, 49) — sum to 1
        context = (weights.unsqueeze(-1) * vals).sum(1)  # (B, H)
        return context, weights

    def forward(self, images, captions):
        visual_feat = self.encode_image(images)      # (B, 49, H)
        # Initialise hidden state as mean of spatial features
        h = visual_feat.mean(1)                      # (B, H)
        c = torch.zeros_like(h)

        logits_all, attn_all = [], []
        for t in range(captions.size(1)):
            word_emb          = self.embed(captions[:, t])   # (B, embed_dim)
            context, attn_w   = self.attend(h, visual_feat)  # (B, H), (B, 49)
            decoder_input     = torch.cat([word_emb, context], dim=1)
            h, c              = self.decoder(decoder_input, (h, c))
            logits_all.append(self.output(h).unsqueeze(1))
            attn_all.append(attn_w.unsqueeze(1))

        logits = torch.cat(logits_all, dim=1)    # (B, seq_len, vocab)
        attn   = torch.cat(attn_all,   dim=1)    # (B, seq_len, 49)
        # attn[:, t, :] reshaped to (7, 7) shows which image region
        # the model attended to when generating word t
        return logits, attn

captioner = ImageCaptioner(vocab_size=5000, embed_dim=256, hidden_dim=512)
images   = torch.randn(2, 3, 224, 224)        # 2 images
captions = torch.randint(0, 5000, (2, 10))    # 10 word captions
logits, attn = captioner(images, captions)
print(f'Output logits:     {logits.shape}')   # (2, 10, 5000)
print(f'Attention maps:    {attn.shape}')     # (2, 10, 49)
print(f'Reshape attn[t]:   {attn[0,0].view(7,7).shape} — spatial heatmap per word')